# Clasificación de severidad de retinopatía diabética: ResNet50 + atención vs. VGG16 vs. baseline

- Objetivo: clasificar el grado de severidad de retinopatía diabética (5 niveles) a partir de imágenes de fondo de ojo, y medir si el accuracy general refleja lo que realmente importa: si el modelo detecta los casos de enfermedad, no solo los sanos.
- Este notebook acompaña al artículo completo, con el detalle de las decisiones de ingeniería y la interpretación honesta del hallazgo principal: **artículo completo →** https://fuzzyfrog.ai/es/ai-lab/proyectos/salud/clasificacion-severidad-retinopatia-diabetica-resnet50-atencion-tensorflow/
- **Dataset:** público, de Kaggle (https://www.kaggle.com/datasets/tanlikesmath/diabetic-retinopathy-resized), sin datos de paciente identificables fuera del propio dataset abierto.
- **Nota importante:** este es un ejercicio técnico/educativo de clasificación de imágenes. No es un dispositivo médico, no está validado clínicamente, y no debe usarse como herramienta de diagnóstico.
- **Nota de reproducibilidad:** se corrigió un bug del notebook original — en la fase de entrenamiento con capas congeladas del modelo ResNet, la llamada de `fit()` apuntaba a una variable de modelo distinta a la construida (`modelo` en vez de `modelo_Resnet`). Corregido aquí; ver la sección de hallazgos para más detalle.


## Diagrama de la arquitectura

`Backbone preentrenado (ResNet50 o VGG16, weights='imagenet')` → `Atención global` → `Atención por categoría` → `GlobalAveragePooling2D` → `Dense(5, softmax)`.

Los dos bloques de atención y el transfer learning se comparan explícitamente contra un baseline sin ninguno de los dos. El diagrama interactivo completo está en el artículo (sección "Diagrama de la solución").

In [ ]:
# Importar librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.io import imread
import os
from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K
from tensorflow.keras.layers import (Conv2D, BatchNormalization, Activation, Multiply,
                                      Lambda, Reshape, GlobalMaxPool2D, AveragePooling2D,
                                      GlobalAveragePooling2D, Dense)
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Model


## Carga de datos

- Dataset público de Kaggle: imágenes de fondo de ojo recortadas y redimensionadas, con etiqueta de severidad (0 a 4) por imagen.
- Cada paciente aporta hasta 2 imágenes (ojo izquierdo y derecho). El split de datos se hace por `ID_paciente`, no por imagen, para evitar que el mismo paciente aparezca en entrenamiento y en prueba a la vez.

In [ ]:
# Descargar el dataset si no se ha descargado (requiere credenciales de Kaggle configuradas)
if not os.path.isdir("resized_train_cropped"):
    os.environ["KAGGLE_CONFIG_DIR"] = "."
    os.system("kaggle datasets download -d tanlikesmath/diabetic-retinopathy-resized")
    os.system('unzip diabetic-retinopathy-resized.zip "resized_train_cropped/*"')
    os.system('unzip diabetic-retinopathy-resized.zip "trainLabels_cropped.csv"')
    os.system("rm diabetic-retinopathy-resized.zip")


In [ ]:
dir_dataset = os.getcwd()
retina_df = pd.read_csv('trainLabels_cropped.csv')
retina_df.drop(retina_df.columns[retina_df.columns.str.contains('unnamed', case=False)], axis=1, inplace=True)

# ID de paciente (a partir del nombre de archivo, sin exponer ningún dato adicional)
retina_df['ID_paciente'] = retina_df['image'].map(lambda x: x.split('_')[0])
retina_df['path'] = retina_df['image'].map(
    lambda x: os.path.join(dir_dataset, "resized_train_cropped/resized_train_cropped", f'{x}.jpeg')
)
retina_df['ojo'] = retina_df['image'].map(lambda x: 1 if x.split('_')[-1] == 'left' else 0)
retina_df.dropna(inplace=True)
retina_df = retina_df[:5000]  # subconjunto para acelerar el ejercicio
retina_df.head()


## Explicación de los datos

- `level`: grado de severidad (0 = sin retinopatía, 4 = proliferativa, la más severa).
- `ID_paciente`: identificador interno del propio dataset público, usado únicamente para no mezclar el mismo paciente entre splits.
- `ojo`: 1 si la imagen es del ojo izquierdo, 0 si es del derecho.

In [ ]:
retina_df[['level', 'ojo']].hist(figsize=(10, 5))
plt.suptitle("Distribución de severidad y de ojo (izquierdo/derecho)")
plt.show()


## Análisis de datos / split

**Por qué importa esta gráfica:** el histograma de severidad muestra el problema central del proyecto antes de entrenar nada. La clase 0 (sin retinopatía) domina el dataset — cualquier modelo que aprenda el atajo de "predecir siempre sano" puede lograr un accuracy alto sin haber aprendido nada útil.

In [ ]:
# Split 70% train / 15% val / 15% test, por paciente (no por imagen), estratificado por severidad
rr_df = retina_df[['ID_paciente', 'level']].drop_duplicates()
train_ids, res_ids = train_test_split(rr_df['ID_paciente'], test_size=0.20, stratify=rr_df['level'])
res_df = retina_df[retina_df['ID_paciente'].isin(res_ids)][['ID_paciente', 'level']].drop_duplicates()
test_ids, valid_ids = train_test_split(res_df["ID_paciente"], test_size=0.50, stratify=res_df["level"])

train_df = retina_df[retina_df['ID_paciente'].isin(train_ids)]
valid_df = retina_df[retina_df['ID_paciente'].isin(valid_ids)]
test_df = retina_df[retina_df['ID_paciente'].isin(test_ids)]

print('Entrenamiento:', train_df.shape[0], '| Validación:', valid_df.shape[0], '| Prueba:', test_df.shape[0])


In [ ]:
train_df[['level', 'ojo']].hist(figsize=(10, 5))
plt.suptitle("Distribución del set de entrenamiento tras el split")
plt.show()


In [ ]:
# Carga de imágenes con augmentation en el set de entrenamiento
train = ImageDataGenerator(horizontal_flip=True, vertical_flip=True, rotation_range=90)
train_batches = train.flow_from_dataframe(train_df, x_col="path", y_col="level",
                                           class_mode="raw", target_size=(512, 512), batch_size=16)

valid = ImageDataGenerator()
valid_batches = valid.flow_from_dataframe(valid_df, x_col="path", y_col="level",
                                           class_mode="raw", target_size=(512, 512),
                                           batch_size=16, shuffle=False)

test = ImageDataGenerator()
test_batches = test.flow_from_dataframe(test_df, x_col="path", y_col="level",
                                         class_mode="raw", target_size=(512, 512),
                                         batch_size=16, shuffle=False)


## Modelado

### 6.1 Bloques de atención

- **Atención global:** resalta qué canales y qué zonas espaciales de la imagen importan más, antes de decidir la clase.
- **Atención por categoría:** genera un mapa de atención específico para cada una de las 5 clases de severidad, porque cada grado se manifiesta con detalles visuales distintos.

In [ ]:
def atencion_global(entrada):
    forma = K.int_shape(entrada)
    x = AveragePooling2D(pool_size=(forma[1], forma[2]))(entrada)
    x = Conv2D(forma[3], 1, padding='same')(x)
    x = Activation('relu')(x)
    x = Conv2D(forma[3], 1, padding='same')(x)
    x = Activation('sigmoid')(x)
    C_A = Multiply()([x, entrada])
    x = Lambda(lambda x: K.mean(x, axis=-1, keepdims=True))(C_A)
    x = Activation('sigmoid')(x)
    S_A = Multiply()([x, C_A])
    return S_A


def atencion_categorias(entrada, num_clases, k):
    forma = K.int_shape(entrada)
    F = Conv2D(k * num_clases, 1, padding='same')(entrada)
    F = BatchNormalization()(F)
    F1 = Activation('relu')(F)
    x = GlobalMaxPool2D()(F1)
    x = Reshape((num_clases, k))(x)
    S = Lambda(lambda x: K.mean(x, axis=-1, keepdims=False))(x)
    x = Reshape((forma[1], forma[2], num_clases, k))(F1)
    x = Lambda(lambda x: K.mean(x, axis=-1, keepdims=False))(x)
    x = Multiply()([S, x])
    M = Lambda(lambda x: K.mean(x, axis=-1, keepdims=True))(x)
    res = Multiply()([entrada, M])
    return res


### 6.2 Función de construcción y entrenamiento reutilizable

Una sola función construye cualquiera de las 3 configuraciones (backbone + atención opcional + pesos preentrenados opcionales), para que la comparación entre ellas sea justa: mismo pipeline de datos, mismo callback, misma estrategia de fine-tuning.

In [ ]:
def construir_modelo(backbone_cls, con_atencion=True, pesos='imagenet'):
    modelo_base = backbone_cls(include_top=False, weights=pesos, input_shape=(512, 512, 3))
    base_sal = modelo_base.output

    if con_atencion:
        x = atencion_global(base_sal)
        base_sal = atencion_categorias(x, num_clases=5, k=5)

    x = GlobalAveragePooling2D()(base_sal)
    salida = Dense(5, activation='softmax')(x)
    modelo = Model(modelo_base.input, salida)
    return modelo, modelo_base


def entrenar_con_fine_tuning(modelo, modelo_base, nombre_checkpoint, epochs_congelado=1, epochs_fino=70):
    """Entrena primero con el backbone congelado, luego con fine-tuning completo a menor learning rate."""
    checkpoint = tf.keras.callbacks.ModelCheckpoint(nombre_checkpoint, monitor="val_loss", save_freq=10)
    red_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.8, patience=3, verbose=1)

    # Fase 1: backbone congelado
    for capa in modelo_base.layers:
        capa.trainable = False
    modelo.compile(optimizer=Adam(0.005, decay=0.00001),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    # NOTA: en el notebook original esta llamada apuntaba por error a una variable
    # de modelo distinta ("modelo" en vez del modelo recién construido). Corregido aquí.
    modelo.fit(train_batches, steps_per_epoch=len(train_batches),
               validation_data=valid_batches, validation_steps=len(valid_batches),
               epochs=epochs_congelado, workers=2, callbacks=[red_lr, checkpoint])

    # Fase 2: fine-tuning completo, learning rate menor
    for capa in modelo_base.layers:
        capa.trainable = True
    modelo.compile(optimizer=Adam(0.0001, decay=0.00001),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    historia = modelo.fit(train_batches, steps_per_epoch=len(train_batches),
                           validation_data=valid_batches, validation_steps=len(valid_batches),
                           epochs=epochs_fino, workers=2, callbacks=[red_lr, checkpoint])
    return historia


### 6.3 Las tres configuraciones

1. **ResNet50 + atención**, con pesos de ImageNet.
2. **VGG16 + atención**, con pesos de ImageNet.
3. **VGG16 sin atención y sin transfer learning** (`weights=None`), como baseline.

In [ ]:
modelo_Resnet, base_resnet = construir_modelo(ResNet50, con_atencion=True, pesos='imagenet')
modelo_Resnet.summary()
# historia_resnet = entrenar_con_fine_tuning(modelo_Resnet, base_resnet, "Modelo_retinopatia_Resnet50.h5")


In [ ]:
modelo_VGG16, base_vgg = construir_modelo(VGG16, con_atencion=True, pesos='imagenet')
modelo_VGG16.summary()
# historia_vgg = entrenar_con_fine_tuning(modelo_VGG16, base_vgg, "Modelo_retinopatia_VGG16.h5")


In [ ]:
# Baseline: VGG16 completo, sin atención y sin pesos preentrenados
modelo_baseline = VGG16(input_shape=(512, 512, 3), include_top=True, weights=None)
modelo_baseline.compile(optimizer=Adam(0.0001, decay=0.00001),
                         loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# historia_baseline = modelo_baseline.fit(train_batches, steps_per_epoch=len(train_batches),
#                                          validation_data=valid_batches, validation_steps=len(valid_batches),
#                                          epochs=70, workers=2)


## Evaluación

**Por qué importa esta comparación:** el accuracy general de cada modelo, por sí solo, no muestra si el modelo detecta enfermedad. El recall por clase sí. Se evalúan los 3 modelos con el mismo set de prueba y se compara accuracy general contra recall en las clases de severidad 1 a 4.

In [ ]:
def evaluar_modelo(modelo, nombre):
    preds = modelo.predict(x=test_batches, steps=len(test_batches), verbose=0)
    y = test_df.iloc[:, 1]

    cm = confusion_matrix(y_true=y, y_pred=np.argmax(preds, axis=-1), normalize='true')
    plt.imshow(cm, interpolation='nearest', cmap="Reds")
    plt.title(f"Matriz de confusión: {nombre}")
    plt.colorbar()
    tick_marks = np.arange(5)
    plt.xticks(tick_marks, range(5)); plt.yticks(tick_marks, range(5))
    plt.ylabel('Etiqueta real'); plt.xlabel('Etiqueta predicha')
    plt.tight_layout()
    plt.show()

    print(classification_report(y, np.argmax(preds, axis=-1), target_names=[str(i) for i in range(5)]))

# evaluar_modelo(modelo_Resnet, "ResNet50 + atención")
# evaluar_modelo(modelo_VGG16, "VGG16 + atención")
# evaluar_modelo(modelo_baseline, "VGG16 baseline (sin atención, sin transfer learning)")


### Resultados obtenidos en la corrida original (documentados para referencia)

| Modelo | Accuracy | Recall clase 1 | Recall clase 2 | Recall clase 3 | Recall clase 4 |
|---|---|---|---|---|---|
| ResNet50 + atención | 0.94 | 0.75 | 0.92 | 0.88 | 0.71 |
| VGG16 + atención | 0.79 | 0.13 | 0.64 | 0.31 | 0.21 |
| VGG16 baseline (sin atención, sin transfer learning) | 0.68 | 0.00 | 0.00 | 0.00 | 0.00 |

El baseline nunca predijo ninguna clase de severidad 1 a 4: su 68% de accuracy es, en la práctica, el porcentaje de pacientes sanos en el set de prueba.

## Hallazgos principales

- **El accuracy general puede ocultar que un modelo no detecta nada.** El baseline (68% de accuracy) tuvo 0% de recall en las 4 clases de severidad de retinopatía: solo aprendió a predecir la clase mayoritaria.
- **Transfer learning + atención no son mejoras incrementales, son la diferencia entre detectar enfermedad y no detectarla.** Con ResNet50 + atención, el recall en las clases severas subió a 0.71-0.92.
- **El backbone importa tanto como la técnica de atención.** Con la misma arquitectura de atención, VGG16 quedó muy por debajo de ResNet50 (0.79 vs. 0.94 de accuracy, y recall notablemente menor en casi todas las clases severas).
- **Un bug de código puede pasar desapercibido en un notebook largo.** Se encontró y corrigió un error donde la fase de entrenamiento con capas congeladas del ResNet apuntaba a la variable de modelo equivocada. Vale la pena auditar activamente estos detalles antes de confiar en un resultado.
- La disciplina real que deja este proyecto: en un problema desbalanceado, reportar solo el accuracy general no es suficiente, y en un contexto médico, esa omisión tiene consecuencias reales.